# Elastic Net Logistic Regression Classifier for Alzheimer's Disease

This notebook implements a comprehensive Elastic Net Logistic Regression classifier for predicting Alzheimer's disease stage (AD, MCI, CN) from biomarker data.

## Features
- Data loading and preprocessing
- Elastic Net Logistic Regression with GridSearchCV
- Cross-validation metric plots (heatmap)
- Comprehensive test set evaluation
- Feature importance and SHAP explainability
- Production-ready with Google Colab integration

**Author:** Claude Code  
**Date:** 2025-11-12

## 1. Setup and Installation

Install required packages (uncomment if needed):

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn openpyxl joblib shap

## 2. Import Libraries

In [ ]:
import os
import sys
import warnings
from pathlib import Path
from typing import Tuple, Dict, Any, List

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize
import joblib
import shap

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configure matplotlib
plt.style.use('default')
%matplotlib inline

print("✓ All libraries imported successfully!")

## 3. Configuration

In [ ]:
# Define feature columns
FEATURE_COLUMNS = [
    'Abeta40', 'Abeta42', 'Abeta ratio', 'Ptau-', 'tau',
    'DHA', 'Formic acid', 'lactoferrin', 'AD7C-NTP'
]

TARGET_COLUMN = 'diagnosis'

# Results directory
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Results directory: {RESULTS_DIR}")
print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Target column: {TARGET_COLUMN}")

## 4. AlzheimerClassifier Class Definition

In [ ]:
class AlzheimerClassifier:
    """
    Elastic Net Logistic Regression classifier for Alzheimer's disease prediction.
    """

    def __init__(self, results_dir: str = 'results'):
        """
        Initialize the classifier.

        Args:
            results_dir: Directory to save results and models
        """
        self.results_dir = Path(results_dir)
        self.results_dir.mkdir(exist_ok=True)

        self.model = None
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        self.grid_search = None
        self.feature_names = FEATURE_COLUMNS
        self.class_names = None

        # For storing results
        self.cv_results = None
        self.test_results = {}

    def log(self, message: str):
        """Print formatted log message."""
        print(f"[INFO] {message}")

    def load_data(self, file_path: str) -> pd.DataFrame:
        """
        Load data from Excel file.

        Args:
            file_path: Path to the Excel file

        Returns:
            Loaded DataFrame
        """
        self.log(f"Loading data from {file_path}...")

        try:
            df = pd.read_excel(file_path, engine='openpyxl')
            self.log(f"Data loaded successfully. Shape: {df.shape}")
            return df
        except Exception as e:
            self.log(f"Error loading data: {str(e)}")
            raise

    def preprocess_data(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Preprocess the data: drop missing diagnosis, encode labels, split and scale.

        Args:
            df: Input DataFrame

        Returns:
            X_train, X_test, y_train, y_test (scaled features and encoded labels)
        """
        self.log("Preprocessing data...")

        # Drop rows with missing diagnosis
        df_clean = df.dropna(subset=[TARGET_COLUMN])
        self.log(f"After dropping missing diagnosis: {df_clean.shape[0]} samples")

        # Extract features and target
        X = df_clean[self.feature_names].values
        y = df_clean[TARGET_COLUMN].values

        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)
        self.class_names = self.label_encoder.classes_
        self.log(f"Classes: {self.class_names}")

        # Stratified train-test split (80/20)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_encoded,
            test_size=0.2,
            stratify=y_encoded,
            random_state=RANDOM_STATE
        )

        self.log(f"Train set: {X_train.shape[0]} samples")
        self.log(f"Test set: {X_test.shape[0]} samples")

        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        self.log("Data preprocessing completed")

        return X_train_scaled, X_test_scaled, y_train, y_test

    def train_model(self, X_train: np.ndarray, y_train: np.ndarray):
        """
        Train Elastic Net Logistic Regression with GridSearchCV.

        Args:
            X_train: Training features
            y_train: Training labels
        """
        self.log("Setting up Elastic Net Logistic Regression with GridSearchCV...")

        # Define base model
        base_model = LogisticRegression(
            multi_class='multinomial',
            solver='saga',
            penalty='elasticnet',
            max_iter=10000,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        # Define hyperparameter grid
        param_grid = {
            'C': [0.01, 0.1, 1, 10],
            'l1_ratio': [0.1, 0.5, 0.9]
        }

        # Setup GridSearchCV with stratified K-fold
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        self.grid_search = GridSearchCV(
            base_model,
            param_grid,
            cv=cv,
            scoring='accuracy',
            n_jobs=-1,
            verbose=1,
            return_train_score=True
        )

        self.log("Running 5-fold stratified cross-validation grid search...")
        self.log(f"Hyperparameter grid: C={param_grid['C']}, l1_ratio={param_grid['l1_ratio']}")
        self.log("This may take a few minutes...")

        # Fit grid search
        self.grid_search.fit(X_train, y_train)

        # Store best model
        self.model = self.grid_search.best_estimator_

        # Store CV results
        self.cv_results = pd.DataFrame(self.grid_search.cv_results_)

        self.log(f"Cross-validation completed!")
        self.log(f"Best parameters: C={self.grid_search.best_params_['C']}, "
                 f"l1_ratio={self.grid_search.best_params_['l1_ratio']}")
        self.log(f"Best CV accuracy: {self.grid_search.best_score_:.4f}")

    def plot_cv_heatmap(self):
        """
        Generate heatmap of cross-validation accuracy across C vs l1_ratio.
        """
        self.log("Generating cross-validation heatmap...")

        # Prepare data for heatmap
        pivot_data = self.cv_results.pivot_table(
            values='mean_test_score',
            index='param_l1_ratio',
            columns='param_C'
        )

        # Create heatmap
        plt.figure(figsize=(10, 6))
        sns.heatmap(
            pivot_data,
            annot=True,
            fmt='.4f',
            cmap='YlGnBu',
            cbar_kws={'label': 'Mean CV Accuracy'},
            linewidths=0.5
        )
        plt.title('Cross-Validation Accuracy Heatmap\n(Elastic Net Logistic Regression)',
                  fontsize=14, fontweight='bold')
        plt.xlabel('C (Inverse Regularization Strength)', fontsize=12)
        plt.ylabel('l1_ratio (L1 vs L2 Balance)', fontsize=12)
        plt.tight_layout()

        # Save
        save_path = self.results_dir / 'cv_heatmap_accuracy.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        self.log(f"Heatmap saved to {save_path}")

        # Optional: Line plots of accuracy vs C for each l1_ratio
        self._plot_cv_line_plots()

    def _plot_cv_line_plots(self):
        """Generate line plots of accuracy vs C for each l1_ratio."""
        plt.figure(figsize=(10, 6))

        for l1_ratio in self.cv_results['param_l1_ratio'].unique():
            subset = self.cv_results[self.cv_results['param_l1_ratio'] == l1_ratio]
            plt.plot(
                subset['param_C'],
                subset['mean_test_score'],
                marker='o',
                label=f'l1_ratio={l1_ratio}',
                linewidth=2
            )

        plt.xscale('log')
        plt.xlabel('C (Inverse Regularization Strength)', fontsize=12)
        plt.ylabel('Mean CV Accuracy', fontsize=12)
        plt.title('Cross-Validation Accuracy vs Regularization Parameter C',
                  fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        save_path = self.results_dir / 'cv_line_plot_accuracy.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        self.log(f"Line plot saved to {save_path}")

    def evaluate_model(self, X_test: np.ndarray, y_test: np.ndarray):
        """
        Evaluate model on test set with comprehensive metrics.

        Args:
            X_test: Test features
            y_test: Test labels
        """
        self.log("Evaluating model on test set...")

        # Predictions
        y_pred = self.model.predict(X_test)
        y_pred_proba = self.model.predict_proba(X_test)

        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision_macro = precision_score(y_test, y_pred, average='macro')
        recall_macro = recall_score(y_test, y_pred, average='macro')
        f1_macro = f1_score(y_test, y_pred, average='macro')

        # Per-class metrics
        precision_per_class = precision_score(y_test, y_pred, average=None)
        recall_per_class = recall_score(y_test, y_pred, average=None)
        f1_per_class = f1_score(y_test, y_pred, average=None)

        # Store results
        self.test_results = {
            'accuracy': accuracy,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'f1_macro': f1_macro,
            'precision_per_class': precision_per_class,
            'recall_per_class': recall_per_class,
            'f1_per_class': f1_per_class,
            'y_test': y_test,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }

        # Print summary
        print("\n" + "="*70)
        print("MODEL EVALUATION RESULTS")
        print("="*70)
        print(f"Model Accuracy: {accuracy:.4f}")
        print(f"Best Params: C={self.grid_search.best_params_['C']}, "
              f"l1_ratio={self.grid_search.best_params_['l1_ratio']}")
        print(f"Macro Precision: {precision_macro:.4f} | "
              f"Macro Recall: {recall_macro:.4f} | "
              f"Macro F1: {f1_macro:.4f}")
        print("\nPer-Class Metrics:")
        for i, class_name in enumerate(self.class_names):
            print(f"  {class_name}: Precision={precision_per_class[i]:.4f}, "
                  f"Recall={recall_per_class[i]:.4f}, F1={f1_per_class[i]:.4f}")
        print("="*70 + "\n")

        # Generate detailed classification report
        self._save_classification_report(y_test, y_pred)

        # Generate visualizations
        self._plot_confusion_matrix(y_test, y_pred)
        self._plot_roc_curves(X_test, y_test)

    def _save_classification_report(self, y_test: np.ndarray, y_pred: np.ndarray):
        """Save detailed classification report."""
        report = classification_report(
            y_test, y_pred,
            target_names=self.class_names,
            digits=4
        )

        report_path = self.results_dir / 'classification_report.txt'
        with open(report_path, 'w') as f:
            f.write("CLASSIFICATION REPORT\n")
            f.write("="*70 + "\n\n")
            f.write(f"Model: Elastic Net Logistic Regression\n")
            f.write(f"Best Parameters: C={self.grid_search.best_params_['C']}, "
                    f"l1_ratio={self.grid_search.best_params_['l1_ratio']}\n")
            f.write(f"Test Accuracy: {self.test_results['accuracy']:.4f}\n\n")
            f.write(report)

        self.log(f"Classification report saved to {report_path}")

        # Also print to console
        print("\nFull Classification Report:")
        print(report)

    def _plot_confusion_matrix(self, y_test: np.ndarray, y_pred: np.ndarray):
        """Plot and save confusion matrix."""
        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(10, 8))
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=self.class_names,
            yticklabels=self.class_names,
            cbar_kws={'label': 'Count'}
        )
        plt.title('Confusion Matrix\n(Elastic Net Logistic Regression)',
                  fontsize=14, fontweight='bold')
        plt.ylabel('True Label', fontsize=12)
        plt.xlabel('Predicted Label', fontsize=12)
        plt.tight_layout()

        save_path = self.results_dir / 'confusion_matrix.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        self.log(f"Confusion matrix saved to {save_path}")

        # Print confusion matrix
        print("\nConfusion Matrix:")
        print(pd.DataFrame(cm, index=self.class_names, columns=self.class_names))

    def _plot_roc_curves(self, X_test: np.ndarray, y_test: np.ndarray):
        """Plot ROC curves (one-vs-rest) with micro and macro averages."""
        # Binarize labels for multi-class ROC
        y_test_bin = label_binarize(y_test, classes=range(len(self.class_names)))
        n_classes = len(self.class_names)

        # Get prediction probabilities
        y_score = self.model.predict_proba(X_test)

        # Compute ROC curve and AUC for each class
        fpr = dict()
        tpr = dict()
        roc_auc = dict()

        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        # Compute micro-average ROC curve and AUC
        fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

        # Compute macro-average ROC curve and AUC
        all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
        mean_tpr /= n_classes
        fpr["macro"] = all_fpr
        tpr["macro"] = mean_tpr
        roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

        # Plot ROC curves
        plt.figure(figsize=(12, 8))

        # Plot micro and macro averages
        plt.plot(
            fpr["micro"], tpr["micro"],
            label=f'Micro-average (AUC = {roc_auc["micro"]:.3f})',
            color='deeppink', linestyle=':', linewidth=3
        )
        plt.plot(
            fpr["macro"], tpr["macro"],
            label=f'Macro-average (AUC = {roc_auc["macro"]:.3f})',
            color='navy', linestyle=':', linewidth=3
        )

        # Plot per-class ROC curves
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
        for i, color in zip(range(n_classes), colors[:n_classes]):
            plt.plot(
                fpr[i], tpr[i],
                color=color,
                lw=2,
                label=f'{self.class_names[i]} (AUC = {roc_auc[i]:.3f})'
            )

        # Plot diagonal
        plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title('ROC Curves - One-vs-Rest\n(Elastic Net Logistic Regression)',
                  fontsize=14, fontweight='bold')
        plt.legend(loc="lower right", fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        save_path = self.results_dir / 'roc_curve.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        self.log(f"ROC curves saved to {save_path}")

        print(f"\nROC AUC Scores:")
        for i, class_name in enumerate(self.class_names):
            print(f"  {class_name}: {roc_auc[i]:.4f}")
        print(f"  Micro-average: {roc_auc['micro']:.4f}")
        print(f"  Macro-average: {roc_auc['macro']:.4f}")

    def show_predictions(self, X_test: np.ndarray, y_test: np.ndarray, n_samples: int = 5):
        """
        Show random prediction examples.

        Args:
            X_test: Test features
            y_test: Test labels
            n_samples: Number of samples to show
        """
        self.log(f"Showing {n_samples} random prediction examples...")

        # Get predictions
        y_pred_proba = self.model.predict_proba(X_test)
        y_pred = self.model.predict(X_test)

        # Select random samples
        indices = np.random.choice(len(X_test), size=min(n_samples, len(X_test)), replace=False)

        print("\n" + "="*70)
        print("PREDICTION EXAMPLES")
        print("="*70)

        for idx in indices:
            true_label = self.class_names[y_test[idx]]
            pred_label = self.class_names[y_pred[idx]]
            probas = y_pred_proba[idx]

            print(f"\nSample {idx}:")
            for i, class_name in enumerate(self.class_names):
                print(f"  {class_name}: {probas[i]:.4f}", end="")
                if i < len(self.class_names) - 1:
                    print(" |", end="")
                else:
                    print()
            print(f"  → Predicted: {pred_label} | True: {true_label}")
            if pred_label == true_label:
                print("  ✓ Correct")
            else:
                print("  ✗ Incorrect")

        print("="*70 + "\n")

    def analyze_feature_importance(self):
        """
        Analyze and plot feature importance using model coefficients.
        """
        self.log("Analyzing feature importance...")

        # Get model coefficients (shape: n_classes x n_features)
        coefficients = self.model.coef_

        # Compute mean absolute coefficient across classes
        feature_importance = np.mean(np.abs(coefficients), axis=0)

        # Create DataFrame
        importance_df = pd.DataFrame({
            'Feature': self.feature_names,
            'Importance': feature_importance
        }).sort_values('Importance', ascending=False)

        # Save to CSV
        csv_path = self.results_dir / 'feature_importance.csv'
        importance_df.to_csv(csv_path, index=False)
        self.log(f"Feature importance saved to {csv_path}")

        # Print top features
        print("\n" + "="*70)
        print("FEATURE IMPORTANCE (Mean Absolute Coefficients)")
        print("="*70)
        print(importance_df.to_string(index=False))
        print("="*70 + "\n")

        # Plot feature importance
        self._plot_feature_importance(importance_df)

        return importance_df

    def _plot_feature_importance(self, importance_df: pd.DataFrame):
        """Plot feature importance bar chart."""
        # Take top 10 features
        top_features = importance_df.head(10)

        plt.figure(figsize=(12, 6))
        bars = plt.barh(
            range(len(top_features)),
            top_features['Importance'],
            color='steelblue'
        )

        # Color the bars with gradient
        colors = plt.cm.Blues(np.linspace(0.4, 0.8, len(top_features)))
        for bar, color in zip(bars, colors):
            bar.set_color(color)

        plt.yticks(range(len(top_features)), top_features['Feature'])
        plt.xlabel('Mean Absolute Coefficient', fontsize=12)
        plt.title('Top 10 Most Important Features\n(Elastic Net Logistic Regression)',
                  fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()

        save_path = self.results_dir / 'feature_importance_plot.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        self.log(f"Feature importance plot saved to {save_path}")

    def explain_with_shap(self, X_train: np.ndarray, X_test: np.ndarray):
        """
        Generate SHAP explanations for model predictions.

        Args:
            X_train: Training features (for background)
            X_test: Test features (for explanation)
        """
        self.log("Generating SHAP explanations (this may take a few minutes)...")

        # Use a sample of training data as background (for efficiency)
        background_size = min(100, len(X_train))
        background = X_train[np.random.choice(X_train.shape[0], background_size, replace=False)]

        # Create SHAP explainer
        explainer = shap.KernelExplainer(self.model.predict_proba, background)

        # Calculate SHAP values for a sample of test data
        test_sample_size = min(50, len(X_test))
        test_sample_indices = np.random.choice(X_test.shape[0], test_sample_size, replace=False)
        X_test_sample = X_test[test_sample_indices]

        self.log(f"Computing SHAP values for {test_sample_size} test samples...")
        shap_values = explainer.shap_values(X_test_sample)

        # Global explanation: Summary plot (beeswarm)
        self._plot_shap_summary(shap_values, X_test_sample)

    def _plot_shap_summary(self, shap_values, X_test_sample: np.ndarray):
        """Plot SHAP summary plot (beeswarm)."""
        # For multi-class, we'll plot for each class
        for i, class_name in enumerate(self.class_names):
            plt.figure(figsize=(12, 8))
            shap.summary_plot(
                shap_values[i] if isinstance(shap_values, list) else shap_values[:, :, i],
                X_test_sample,
                feature_names=self.feature_names,
                show=False,
                plot_size=(12, 8)
            )
            plt.title(f'SHAP Summary Plot - {class_name}\n(Feature Impact on Model Output)',
                      fontsize=14, fontweight='bold')
            plt.tight_layout()

            save_path = self.results_dir / f'shap_summary_plot_{class_name}.png'
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()

            self.log(f"SHAP summary plot for {class_name} saved to {save_path}")

    def save_model(self):
        """Save trained model and preprocessing objects."""
        self.log("Saving model and preprocessing objects...")

        # Save model
        model_path = self.results_dir / 'elasticnet_model.pkl'
        joblib.dump({
            'model': self.model,
            'scaler': self.scaler,
            'label_encoder': self.label_encoder,
            'feature_names': self.feature_names,
            'class_names': self.class_names,
            'best_params': self.grid_search.best_params_,
            'best_score': self.grid_search.best_score_
        }, model_path)

        self.log(f"Model saved to {model_path}")

        # Print reload example
        print("\n" + "="*70)
        print("MODEL PERSISTENCE")
        print("="*70)
        print(f"Model saved to: {model_path}")
        print("\nTo reload the model, use:")
        print("-" * 70)
        print("import joblib")
        print(f"model_data = joblib.load('{model_path}')")
        print("model = model_data['model']")
        print("scaler = model_data['scaler']")
        print("label_encoder = model_data['label_encoder']")
        print("\n# Make predictions on new data:")
        print("X_new_scaled = scaler.transform(X_new)")
        print("predictions = model.predict(X_new_scaled)")
        print("predicted_labels = label_encoder.inverse_transform(predictions)")
        print("="*70 + "\n")

print("✓ AlzheimerClassifier class defined successfully!")

## 5. File Upload (Google Colab)

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab")

# Upload file if in Colab
if IN_COLAB:
    from google.colab import files
    print("\nPlease upload your Excel file:")
    uploaded = files.upload()
    INPUT_FILE = list(uploaded.keys())[0]
    print(f"\n✓ File uploaded: {INPUT_FILE}")
else:
    # If not in Colab, specify your file path here
    INPUT_FILE = 'your_data_file.xlsx'  # Change this to your file path
    print(f"Using file: {INPUT_FILE}")

## 6. Initialize Classifier

In [ ]:
# Initialize the classifier
classifier = AlzheimerClassifier(results_dir='results')
print("✓ Classifier initialized!")

## 7. Load and Explore Data

In [ ]:
# Load data
df = classifier.load_data(INPUT_FILE)

# Display basic information
print("\nDataset Info:")
print(df.info())

print("\nFirst few rows:")
display(df.head())

print("\nClass distribution:")
print(df[TARGET_COLUMN].value_counts())

## 8. Preprocess Data

In [ ]:
# Preprocess data
X_train, X_test, y_train, y_test = classifier.preprocess_data(df)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Number of classes: {len(classifier.class_names)}")
print(f"Class names: {classifier.class_names}")

## 9. Train Model with Cross-Validation

In [ ]:
# Train the model with grid search cross-validation
classifier.train_model(X_train, y_train)

## 10. Visualize Cross-Validation Results

In [ ]:
# Plot cross-validation heatmap and line plots
classifier.plot_cv_heatmap()

## 11. Evaluate Model on Test Set

In [ ]:
# Evaluate model on test set
classifier.evaluate_model(X_test, y_test)

## 12. Show Prediction Examples

In [ ]:
# Show some prediction examples
classifier.show_predictions(X_test, y_test, n_samples=10)

## 13. Feature Importance Analysis

In [ ]:
# Analyze feature importance
importance_df = classifier.analyze_feature_importance()

# Display as a nice table
display(importance_df)

## 14. SHAP Explainability (Optional - Takes Time)

In [ ]:
# Generate SHAP explanations (this may take several minutes)
# Uncomment the line below to run SHAP analysis
# classifier.explain_with_shap(X_train, X_test)

## 15. Save Model

In [ ]:
# Save the trained model and preprocessing objects
classifier.save_model()

## 16. Model Loading Example

In [ ]:
# Example: Load the saved model and make predictions on new data
import joblib

# Load model
model_data = joblib.load('results/elasticnet_model.pkl')
loaded_model = model_data['model']
loaded_scaler = model_data['scaler']
loaded_label_encoder = model_data['label_encoder']

print("✓ Model loaded successfully!")
print(f"\nModel details:")
print(f"  Best parameters: {model_data['best_params']}")
print(f"  Best CV score: {model_data['best_score']:.4f}")
print(f"  Feature names: {model_data['feature_names']}")
print(f"  Class names: {model_data['class_names']}")

# Example prediction on test set
X_test_scaled = loaded_scaler.transform(X_test)
predictions = loaded_model.predict(X_test_scaled)
predicted_labels = loaded_label_encoder.inverse_transform(predictions)

print(f"\n✓ Made {len(predictions)} predictions successfully!")

## 17. Summary Statistics

In [ ]:
# Print final summary
print("\n" + "="*70)
print("PIPELINE COMPLETED SUCCESSFULLY!")
print("="*70)
print(f"✓ Model accuracy: {classifier.test_results['accuracy']:.4f}")
print(f"✓ Best parameters: C={classifier.grid_search.best_params_['C']}, "
      f"l1_ratio={classifier.grid_search.best_params_['l1_ratio']}")
print(f"✓ Macro F1-score: {classifier.test_results['f1_macro']:.4f}")
print(f"✓ All results saved to: {classifier.results_dir}/")
print("\nGenerated files:")
for file in sorted(classifier.results_dir.glob('*')):
    print(f"  - {file.name}")
print("="*70)